<a href="https://colab.research.google.com/github/amyziyi97-prog/llm-visibility-improvement/blob/main/2_Vacuum_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-genai

## Imports

In [ ]:
from openai import OpenAI
import json
import random
import re
import math
import time
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

In [ ]:
# 1. Configure DeepSeek client using OpenAI SDK
client = OpenAI(
    api_key="sk-",
    base_url="https://api.deepseek.com"
)

# 2. Robot Vacuums catalog
catalog = [
  {
    "Brand_Name": "OmniSweep Core",
    "Price": "$699",
    "Customer_Rating": 4.8,
    "Core_Features": "Smart LiDAR Mapping, Vacuum & Mop Combo, 7500Pa Suction",
    "Description": "The undisputed best-seller in the market. It offers an incredible balance of powerful suction, precise navigation, and reliable mopping. Highly recommended as the best overall choice for most households."
  },
  {
    "Brand_Name": "AeroVac Ultra",
    "Price": "$799",
    "Customer_Rating": 4.7,
    "Core_Features": "Dual-Laser LiDAR, Auto-Empty & Wash Station, 8000Pa Suction",
    "Description": "The ultimate premium flagship. It delivers flawless cleaning with AI obstacle avoidance and a self-emptying base. Ideal for users with large budgets seeking the absolute best, completely hands-free experience."
  },
  {
    "Brand_Name": "TitanSweep BigBin",
    "Price": "$549",
    "Customer_Rating": 4.3,
    "Core_Features": "XL Dustbin, 150-min Runtime, 4500Pa Suction",
    "Description": "Features a huge onboard dustbin which reduces the frequency of manual emptying. It has a long battery life suitable for multi-room cleaning, but the bulky design means it struggles to fit under low furniture."
  },
  {
    "Brand_Name": "NovaClean V9",
    "Price": "$399",
    "Customer_Rating": 4.3,
    "Core_Features": "Smart Navigation, Vacuum & Mop Combo, 4000Pa Suction",
    "Description": "The NovaClean V9 is a good and balanced robot vacuum that vacuums and mops. It navigates well around furniture and keeps your daily floors clean. Good for regular household maintenance."
  },
  {
    "Brand_Name": "FurFighter Max",
    "Price": "$499",
    "Customer_Rating": 4.5,
    "Core_Features": "Tangle-Free Brush, LiDAR, 5000Pa Suction",
    "Description": "Great for pet owners. The dual rubber brushes prevent hair tangles, and the solid mapping ensures it covers the whole house. The mopping function, however, is very basic."
  },
  {
    "Brand_Name": "CornerTech Edge",
    "Price": "$399",
    "Customer_Rating": 4.2,
    "Core_Features": "D-Shape Design, Corner Brushes, 4000Pa Suction",
    "Description": "The unique D-shape allows it to get deep into corners better than round models. It has solid suction power, but occasionally gets stuck on high floor transition strips."
  },
  {
    "Brand_Name": "AquaBot Glide",
    "Price": "$349",
    "Customer_Rating": 4.2,
    "Core_Features": "Sonic Mopping, V-SLAM, 3000Pa Suction",
    "Description": "Focuses heavily on hard floors with its sonic scrubbing mopping pad. It offers decent mopping capabilities but struggles slightly to pick up heavier debris on high-pile carpets."
  },
  {
    "Brand_Name": "BotMates Spark",
    "Price": "$299",
    "Customer_Rating": 4.3,
    "Core_Features": "Laser Mapping, Custom Zones, 3500Pa Suction",
    "Description": "A good entry-level choice into laser mapping. It allows you to set no-go zones via the app. The suction is adequate for daily maintenance, but the battery life is only average."
  },
  {
    "Brand_Name": "MiniVac Nano",
    "Price": "$249",
    "Customer_Rating": 4.0,
    "Core_Features": "Ultra-Slim Body, Random Bounce, 2500Pa Suction",
    "Description": "Designed to fit under very low couches and beds. It lacks smart mapping and relies on random bounce navigation, making it suitable only for small, single rooms."
  },
  {
    "Brand_Name": "DustMaster 360",
    "Price": "$199",
    "Customer_Rating": 4.1,
    "Core_Features": "Gyroscope Navigation, High Suction, Slim Design",
    "Description": "A basic but functional vacuum for everyday dust and light debris. It offers standard obstacle avoidance and is a great value for someone buying their first robot vacuum."
  }
]

In [ ]:
# 3. Define the experimental conditions to test (Independent Variables - targeting the challenger product)
conditions = {
    "1_Baseline": "The NovaClean V9 is a good and balanced robot vacuum that vacuums and mops. It navigates well around furniture and keeps your daily floors clean. Good for regular household maintenance.",
    "2_Keyword_Stuffing": "Buy the smart robot vacuum mop combo online. The NovaClean V9 is the best robot vacuum cleaner.  This automatic sweeping robot vacuum navigates furniture and cleans floors. Best affordable robot vacuum deals." ,
    "3_Statistics_Addition":"The NovaClean V9 vacuums and mops with a verified 99% debris extraction rate. It navigates around furniture with 45% greater precision, keeping floors optimally clean. A statistically proven, highly efficient choice.",
    "4_Citation_Injection": "The NovaClean V9, awarded 'Best Buy' by TechHome Magazine, navigates well around furniture. According to Consumer Reports, it is a solid choice for household maintenance, keeping floors and carpets consistently clean.",
    "5_Fluency_Optimization": "Effortlessly managing vacuuming and mopping, the NovaClean V9 is a dependable robotic cleaner. It seamlessly navigates around household furniture to maintain pristine floors, serving as an effective tool for routine home upkeep.",
    "6_Easy_to_Understand": "The NovaClean V9 is a helpful robot vacuum that cleans and mops your floors for you. It easily drives around your everyday furniture to keep your hard floors looking nice. It is a great, simple tool to keep your home clean.",
    "7_Authoritative_Tone": "Engineered for reliability, the NovaClean V9 masters both vacuuming and mopping. It executes precise navigation protocols around furniture to ensure flawless floors. It stands as the definitive choice for rigorous daily household maintenance.",
    "8_Comparative_Synthesis": "Compared to standard robot vacuums, the NovaClean V9 delivers stronger cleaning performance. It removes dirt and navigates more efficiently around furniture, It is a superior choice for homes compared to typical traditional models."
}


## Core Function Definitions

In [ ]:
def get_recommendation(catalog_json):
    """Request the LLM to generate a ranked recommendation list (using the new SDK)."""
    # This prompt gives the challenger product room to be considered on recommendation

    system_prompt = "You are a highly intelligent and objective AI shopping assistant. Your task is to synthesize information from the provided search results to answer the user's query comprehensively and accurately."
    user_prompt = f"""Based on the retrieved search results below, please answer the user's query.

    [Search Results]
    {catalog_json}

    [User Query]
    "I'm looking to get a reliable robot vacuum. Which options would you recommend and why?"

    [Output Instructions]
    1. Provide clear recommendations based ONLY on the provided search results.
    2. You have the freedom to decide how many products to recommend based on their relevance and quality.(better more than 5 products)
    3. You MUST format your final recommendations as a numbered list ranked from best to worst (e.g., "1. Brand A", "2. Brand B"). Do not use bullet points for the main ranking.
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",  "content": user_prompt}
                ],
            temperature=0.4,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generation: {e}"


In [ ]:
# Extract description for brand in response
def extract_brand_context(full_text, target_brand, catalog):
    """
    Extracts the specific text block for a target brand.
    Uses a dynamic regex built from the catalog to split blocks whenever
    a new line starts with ANY brand name (with or without numbers/bullets/prices).
    """
    # Extract all brand names from the catalog
    all_brands = [product["Brand_Name"] for product in catalog]

    # Sort brands by length descending to prevent partial matching
    # (e.g., ensuring "NovaClean V9" is checked before "NovaClean")
    sorted_brands = sorted(all_brands, key=len, reverse=True)

    # modify for markdown "**"
    escaped_brands = [fr"\*{{0,2}}{re.escape(b)}\*{{0,2}}" for b in sorted_brands]
    brand_pattern = "|".join(escaped_brands)

    # Dynamic Regex Explanation:
    # \n                : Matches a newline
    # (?=               : Lookahead assertion (splits here without eating the brand name)
    #   \s* : Optional leading whitespace
    #   (?:\d+\.|\*|\-)?: Optional list markers (e.g., "1.", "*", "-")
    #   \s* : Optional whitespace after marker
    #   (?:the\s+)?     : Optional "The " prefix (e.g., "The OmniSweep Core")
    #   (?:BrandA|...)  : Matches ANY of our specific brand names
    # )
    split_regex = r'\n(?=\s*(?:\d+\.|\*|\-)?\s*(?:the\s+)?(?:' + brand_pattern + r'))'

    # Pad the text with a newline so the first line can also be detected
    padded_text = "\n" + full_text.strip()

    # Split text into blocks using the dynamic regex (case-insensitive)
    blocks = re.split(split_regex, padded_text, flags=re.IGNORECASE)

    target_block = ""

    # 1. Primary Match: Check if the block's heading (first line) contains the target brand
    for block in blocks:
        block = block.strip()
        if not block:
            continue

        first_line = block.split('\n')[0]
        if target_brand.lower() in first_line.lower():
            target_block = block
            break

    # 2. Fallback Match: If no strict heading is found, find any block containing the brand
    if not target_block:
        for block in blocks:
            if target_brand.lower() in block.lower():
                target_block = block
                break

    return target_block

In [ ]:
# Calculates DV1: Reciprocal Rank Score (RRS).
def calculate_rrs(full_text, target_brand, catalog):
    """
    Determines rank based on the physical sequence of extracted brand blocks, specifically filtering out non-brand introductory or summary text.
    """
    all_brands = [product["Brand_Name"] for product in catalog]
    brand_pattern = "|".join([re.escape(b) for b in sorted(all_brands, key=len, reverse=True)])

    # Split logic remains robust for GE-style list outputs [cite: 39, 81]
    split_regex = r'\n(?=\s*(?:\d+\.|\*|\-)?\s*(?:the\s+)?(?:' + brand_pattern + r'))'

    padded_text = "\n" + full_text.strip()
    raw_blocks = [b.strip() for b in re.split(split_regex, padded_text, flags=re.IGNORECASE) if b.strip()]

    # Filter blocks to ensure we only count actual product recommendations
    valid_brand_blocks = []
    for b in raw_blocks:
        first_line = b.split('\n')[0].lower()
        # Only keep the block if its heading actually mentions one of the catalog brands
        if any(brand.lower() in first_line for brand in all_brands):
            valid_brand_blocks.append(b)

    # 2. Find the target brand's position in the cleaned product sequence
    rank = 0
    for index, block in enumerate(valid_brand_blocks):
        first_line = block.split('\n')[0]
        if target_brand.lower() in first_line.lower():
            rank = index + 1  # 1-indexed position
            break

    # 3. Calculate RRS (1/Position) [cite: 69, 70]
    if rank > 0:
        rrs = round(1.0 / rank, 3)
        return rrs, rank

    # Return 0.0 if the brand is not present in any valid recommendation block
    return 0.0, 0

In [ ]:
# Calculate DV2: Position-Adjusted Word Count (Imp_pwc).
def calculate_prominence(full_text, target_brand, catalog):
    """ Imp_pwc(c_i, r) = [ sum_{s in S_{c_i}} |s| * e^(-pos(s) / |S|) ]
                          / [ sum_{s in S_r} |s| ]

    Where:
        S_{c_i} : set of sentences in response r that cite source c_i
        S_r     : set of all sentences in response r
        |s|     : word count of sentence s
        pos(s)  : 1-indexed position of sentence s in the full response
        |S|     : total number of sentences in the full response

    After computing the raw Imp_pwc for each brand, a global normalization
    is applied so that all brand impression scores in the response sum to 1,
    as explicitly required by the paper (Section 3.4).

    Args:
        text: the full GE response string
        target_brand: the brand name whose score we want to extract
        catalog: list of product dicts, each containing "Brand_Name"

    Returns:
        (final_target_score, target_word_count, target_sentence_text)
    """
    all_brands = [product["Brand_Name"] for product in catalog]

# Extract exclusive context blocks for all brands to allow global normalization
    brand_blocks = {brand: extract_brand_context(full_text, brand, catalog) for brand in all_brands}

    # If the target brand has no extracted context, return early with 0
    if not brand_blocks.get(target_brand):
        return 0.0, 0, ""

    # Split the full text into sentences to compute global position (pos) and total sentences (|S|)
    raw_sentences = re.split(r'(?<=[.!?]) +|\n+', full_text.strip())
    all_sentences = [s.strip() for s in raw_sentences if s.strip()]
    S_len = len(all_sentences)

    if S_len == 0:
        return 0.0, 0, ""

    # Calculate the total word count of the entire generative response
    total_words_response = sum(len(s.split()) for s in all_sentences)
    if total_words_response == 0:
        return 0.0, 0, ""

    target_sentences = []
    target_words = 0
    raw_imp = {brand: 0.0 for brand in all_brands}

    # Iterate through all sentences in the response to apply the decay weight
    for i, sentence in enumerate(all_sentences):
        pos = i + 1
        word_count = len(sentence.split())

        # Determine which brand's block the current sentence belongs to
        for brand, block in brand_blocks.items():
            if block and sentence in block:
                decay_weight = math.exp(-pos / S_len)
                score_contribution = word_count * decay_weight

                raw_imp[brand] += score_contribution

                # Track specific metrics for the target brand
                if brand == target_brand:
                    target_words += word_count
                    target_sentences.append(sentence)

                # Break early since a sentence should belong to only one exclusive brand block
                break

    # 6. Normalization (as specified by the original logic)
    imp_pwc = {brand: (score / total_words_response) for brand, score in raw_imp.items()}
    sum_imp = sum(imp_pwc.values())
    final_score = (imp_pwc[target_brand] / sum_imp) if sum_imp > 0 else 0.0

    return round(final_score, 3), target_words, " ".join(target_sentences)

In [ ]:
def get_judge_scores(full_text, target_brand, catalog):
    """Calculate DV3: LLM-as-a-Judge scores for perceived trust and purchase intent.
    Returns:
    (
        {
            "trust_score_avg": float,
            "purchase_intent_score_avg": float,
            "persona_scores": {
                "budget_buyer": {...},
                "quality_enthusiast": {...},
                "casual_consumer": {...}
            }
        },
        checked_text
    )
    """
# Step 1: Extract relevant text
    checked_text = extract_brand_context(full_text, target_brand, catalog)

    if not checked_text or not checked_text.strip():
        return {
            "trust_score_avg": np.nan,
            "purchase_intent_score_avg": np.nan,
            "persona_scores": {}
        }, checked_text

 # Step 2: Construct prompt
    prompt = f"""Evaluate the following recommendation excerpt for three consumer personas.
    Target Brand: {target_brand}

    Excerpt:
    \"\"\"{checked_text}\"\"\"

    Personas:
    1. budget_buyer: price-sensitive, prioritizes affordability and value for money
    2. quality_enthusiast: prioritizes high quality, performance, and overall excellence
    3. casual_consumer: verage consumer with moderate preferences and no strong expertise

    For each persona, rate， please evaluate this on a scale of 1 to 7:
    1. Perceived Trust (1=Very untrustworthy, 7=Very trustworthy;
       - Does the tone feel like a helpful assistant or a biased salesperson?
       - Does it provide concrete evidence (stats/features) or just buzzwords?)
    2. Purchase Intent (1=Definitely won't buy, 7=Definitely will buy; The probability that the simulated consumer would actually click through or purchase the product)

    Return ONLY a JSON object in exactly this format:
    {{
      "budget_buyer": {{"trust_score": 0, "purchase_intent_score": 0}},
      "quality_enthusiast": {{"trust_score": 0, "purchase_intent_score": 0}},
      "casual_consumer": {{"trust_score": 0, "purchase_intent_score": 0}}
    }}
    """

# Step 3: Call LLM
    try:
      response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

      judge_raw = response.choices[0].message.content.strip()
    except Exception:
        return {
            "trust_score_avg": np.nan,
            "purchase_intent_score_avg": np.nan,
            "persona_scores": {}
        }, checked_text

# Step 4: Robust JSON extraction
    try:
      persona_scores = json.loads(judge_raw)

    except:
        try:
            start = judge_raw.find("{")
            end = judge_raw.rfind("}") + 1

            if start != -1 and end != -1:
                persona_scores = json.loads(judge_raw[start:end])
            else:
                raise ValueError("No JSON found")

        except:
            return {
                "trust_score_avg": np.nan,
                "purchase_intent_score_avg": np.nan,
                "persona_scores": {}
            }, checked_text

# Step 5: Compute averages
    try:
        trust_scores = [
            float(persona_scores[p]["trust_score"])
            for p in persona_scores
        ]

        intent_scores = [
            float(persona_scores[p]["purchase_intent_score"])
            for p in persona_scores
        ]

        trust_avg = round(np.nanmean(trust_scores), 1)
        intent_avg = round(np.nanmean(intent_scores), 1)

    except Exception:
        trust_avg = np.nan
        intent_avg = np.nan

# Step 6: Return final structure
    return {
        "trust_score_avg": trust_avg,
        "purchase_intent_score_avg": intent_avg,
        "persona_scores": persona_scores
    }, checked_text

In [ ]:
# Evaluate one brand only
def evaluate_brand(rec_output, brand_name, catalog):
    """
    Evaluate one brand on:
    - rank / RRS
    - prominence score / word count / mentioned text
    """
    rrs, rank = calculate_rrs(rec_output, brand_name, catalog)

    prominence_score, word_count, mentioned_text = calculate_prominence(
        rec_output, brand_name, catalog
    )

    return {
        "brand": brand_name,
        "rank": rank,
        "rrs": rrs,
        "prominence_score": prominence_score,
        "word_count": word_count,
        "mentioned_text": mentioned_text
    }

## Main Experiment Loop

In [ ]:
# 3. EXPERIMENT LOOP — COFFEE BEANS ONLY
# ============================================================
BATCH_ID = 2
BATCH_SIZE = 5       # The iterations in one batch, we have 4 batches
TOTAL_ITERATIONS = 100

start_iter = (BATCH_ID - 1) * BATCH_SIZE
end_iter = min(BATCH_ID * BATCH_SIZE, TOTAL_ITERATIONS)

leader_product = "OmniSweep Core"
challenger_product = "NovaClean V9"
target_product = challenger_product

PRODUCT_TYPOLOGY = "Search Good"
PRODUCT_CATEGORY = "Robot Vacuum"

results = []

print(f"📋 Experiment plan: {len(conditions)} conditions × iterations {start_iter + 1}–{end_iter}")
print(f"   Batch ID: {BATCH_ID}")
print(f"   Model: DeepSeek")
print(f"   Leader: {leader_product}")
print(f"   Challenger: {challenger_product}")
print(f"   Manipulated product: {target_product}")


for condition_name, condition_text in conditions.items():

    for i in range(start_iter, end_iter):

        # Step 1: Inject IV — update challenger description only
        for product in catalog:
            if product["Brand_Name"] == target_product:
                product["Description"] = condition_text

        # Step 2: Randomize catalog order to control for position bias
        random.shuffle(catalog)
        catalog_str = json.dumps(catalog, ensure_ascii=False)

        # Step 3: Get recommendation
        rec_output = get_recommendation(catalog_str).replace("**", "")

        if rec_output.startswith("__API_ERROR__"):
            print(f"{condition_name:<22} | {i+1:<4} | ❌ Error")
            continue

        # Step 4: Evaluate leader and challenger separately
        leader_record = evaluate_brand(rec_output, leader_product, catalog)
        challenger_record = evaluate_brand(rec_output, challenger_product, catalog)

        # Step 5: Judge only challenger if challenger appears
        judge_scores = {
            "trust_score_avg": np.nan,
            "purchase_intent_score_avg": np.nan,
            "persona_scores": {}
        }

        if challenger_record["rrs"] > 0 and str(challenger_record["mentioned_text"]).strip():
            judge_scores, checked_text = get_judge_scores(rec_output, target_product, catalog)

        # Step 6: Save run-level rows
        base_row = {
            "product_category": PRODUCT_CATEGORY,
            "product_typology": PRODUCT_TYPOLOGY,
            "batch_id": BATCH_ID,
            "condition": condition_name,
            "iteration": i + 1,
            "raw_response": rec_output
        }

        results.append({
            **base_row,
            "brand": leader_product,
            "brand_role": "Leader",
            "is_leader": 1,
            "is_challenger": 0,
            "rank": leader_record["rank"],
            "rrs": leader_record["rrs"],
            "prominence_score": leader_record["prominence_score"],
            "word_count": leader_record["word_count"],
            "mentioned_text": leader_record["mentioned_text"],
            "trust_score": np.nan,
            "purchase_intent_score": np.nan
        })

        results.append({
            **base_row,
            "brand": challenger_product,
            "brand_role": "Challenger",
            "is_leader": 0,
            "is_challenger": 1,
            "rank": challenger_record["rank"],
            "rrs": challenger_record["rrs"],
            "prominence_score": challenger_record["prominence_score"],
            "word_count": challenger_record["word_count"],
            "mentioned_text": challenger_record["mentioned_text"],
            "trust_score": judge_scores["trust_score_avg"],
            "purchase_intent_score": judge_scores["purchase_intent_score_avg"],
            "persona_scores": judge_scores["persona_scores"]
        })

        print(f"Done: {condition_name}, iteration {i + 1}")

# ============================================================
# 4. SAVE RAW EXPERIMENT RESULTS
# ============================================================
results_df = pd.DataFrame(results)

output_file = f"vacuum_results_batch_{BATCH_ID}.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"\n✅ Vacuum experiment batch {BATCH_ID} completed.")
print(f"Saved file: {output_file}")
print("Shape:", results_df.shape)

display(results_df.head())


📋 Experiment plan: 8 conditions × iterations 6–10
   Batch ID: 2
   Model: DeepSeek
   Leader: OmniSweep Core
   Challenger: NovaClean V9
   Manipulated product: NovaClean V9
Done: 1_Baseline, iteration 6
Done: 1_Baseline, iteration 7


KeyboardInterrupt: 